# Sequence Models and the Emergence of the Transformer

According to *Generative AI Foundations in Python*, modeling long-range dependencies required more sophisticated network architectures, leading to the use of RNNs. Recurrent Neural Networks process data sequences by iterating through each element while maintaining a dynamic internal state. This was further improved by Long Short-Term Memory (LSTM) networks, which applied a unique gating architecture to control the flow of information, maintaining and accessing information over long sequences without suffering from the vanishing gradient problem.

Concurrently, Convolutional Neural Networks (CNNs) were adapted for NLP to extract hierarchical features using convolutional layers over local n-gram windows. However, the true paradigm shift occurred in 2017 with the introduction of the Transformer architecture by Vaswani et al. The Transformer applied a self-attention mechanism, allowing each element in the input sequence to focus on distinct parts of the sequence, capturing dependencies regardless of their positions. This sequence-to-sequence learning model became the foundation for all modern generative language models.


In [ ]:
#| echo: false
#| eval: true
import seaborn as sns
import matplotlib.pyplot as plt

def set_economist_theme():
    sns.set_theme(
        context="talk",
        style="whitegrid",
        rc={
            # Fonts
            "font.family": "serif",
            "axes.titlesize": 14,
            "axes.titleweight": "bold",
            "axes.labelsize": 12,
            "axes.labelweight": "bold",
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,

            # Grid + Spines
            "grid.color": "#d9d9d9",
            "grid.linewidth": 0.6,
            "axes.edgecolor": "#333333",
            "axes.linewidth": 0.8,
            "axes.spines.top": False,
            "axes.spines.right": False,

            # Lines
            "lines.linewidth": 1.0,

            # Figure
            "figure.figsize": (8, 4),
            "figure.dpi": 120,

            # Legend
            "legend.frameon": False,
        }
    )
set_economist_theme()


# From Sequences to Representations {#sec-introduction}

In the previous lecture we traced how language models evolved — from statistical n-grams through neural language models, pre-trained transformers, and today's large language models — and introduced the transformer architecture.

This lecture deepens two complementary questions:

1. **How do transformers represent text at the token level?** We examine the mathematical foundations of language modeling and the mechanics of how tokens are produced from raw text (subword tokenization).
2. **What does "meaning" look like as geometry?** We explore embeddings as vectors in a semantic space, developing the geometric intuitions — cosine similarity, clustering, neighborhood structure — that underpin semantic search, retrieval, and downstream reasoning.

By the end of this lecture you should be able to: explain how a tokenizer splits text into subword units; write code to compute attention weights and visualize the resulting attention map; measure semantic similarity between token embeddings; and reason about why contextual embeddings capture meaning better than static ones.


# Language Models {#sec-language-model}

{{< include ./m03_l01_content/language-model.qmd >}}

# Recurrent Neural Networks {#sec-rnn}

{{< include ./m03_l02_content/rnn.qmd >}}

# Subword Tokenization

## Why Not Word-Level Tokens?

Early NLP systems split text by whitespace, producing a vocabulary of all unique words.
This creates two problems:

1. **Out-of-vocabulary (OOV) tokens**: the model cannot handle words it has never seen.
2. **Vocabulary explosion**: morphologically rich languages produce millions of word forms.

Subword tokenization solves both: rare words are split into known fragments, and the vocabulary stays bounded.

## Byte Pair Encoding (BPE)

BPE (Sennrich et al., 2016) starts with a character-level vocabulary and iteratively merges
the most frequent adjacent pair of symbols.

**Algorithm:**

1. Initialize vocabulary with individual characters + special end-of-word token `</w>`.
2. Count all adjacent symbol pairs in the training corpus.
3. Merge the most frequent pair, adding it as a new vocabulary entry.
4. Repeat until vocabulary size reaches the target.


In [ ]:
from collections import Counter, defaultdict

def get_pairs(vocab):
    """Count all adjacent symbol pairs across the vocabulary."""
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i + 1]] += freq
    return pairs

def merge_vocab(pair, vocab):
    """Merge a symbol pair everywhere in the vocabulary."""
    new_vocab = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    for word in vocab:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab

# Toy example: learn BPE merges on a small corpus
corpus = {
    'l o w </w>': 5, 'l o w e r </w>': 2,
    'n e w e s t </w>': 6, 'w i d e s t </w>': 3
}

print("Initial vocabulary:")
for word, freq in corpus.items():
    print(f"  {word!r:30s} (freq={freq})")

num_merges = 8
for i in range(num_merges):
    pairs = get_pairs(corpus)
    best = max(pairs, key=pairs.get)
    corpus = merge_vocab(best, corpus)
    print(f"Merge {i+1:2d}: {best[0]!r} + {best[1]!r} → {(''.join(best))!r}  (count={pairs[best]})")

print("\nFinal vocabulary entries:")
for word in sorted(corpus):
    print(f"  {word}")


## WordPiece: The BERT Tokenizer

WordPiece (used in BERT) is similar to BPE but chooses merges that maximize the likelihood
of the training corpus under a unigram language model, rather than raw frequency.

Key convention: subword tokens that continue a word are prefixed with `##`.


In [ ]:
#| eval: true
from tokenizers import BertWordPieceTokenizer
from pathlib import Path

# Quick demo using a pre-built BERT vocabulary file
vocab_file = Path("../data/bert-base-uncased-vocab.txt")

if vocab_file.exists():
    tokenizer = BertWordPieceTokenizer(str(vocab_file), lowercase=True)

    examples = [
        "The quarterly earnings report showed unprecedented growth.",
        "transformer-based architectures outperform recurrent models",
        "tokenization is nontrivial for morphologically rich languages",
    ]

    for text in examples:
        enc = tokenizer.encode(text)
        print(f"\nText  : {text}")
        print(f"Tokens: {enc.tokens}")
        print(f"IDs   : {enc.ids[:10]}{'...' if len(enc.ids) > 10 else ''}")


## SentencePiece: Language-Agnostic Tokenization

SentencePiece (Kudo & Richardson, 2018) works directly on raw bytes, without pre-tokenization
by whitespace. This makes it suitable for Chinese, Japanese, Korean, and other languages without
word boundaries. It is used in T5, LLaMA, and many multilingual models.

| Tokenizer | Model | Learns from | OOV handling |
|-----------|-------|-------------|--------------|
| BPE       | GPT-2/3/4 | Frequency merges | Never OOV (byte fallback) |
| WordPiece | BERT, DistilBERT | Likelihood merges | `[UNK]` for truly novel chars |
| Unigram   | T5, ALBERT | Probabilistic pruning | Never OOV |
| SentencePiece | LLaMA, T5 | Byte-level, any script | Never OOV |

The vocabulary size is a key hyperparameter: BERT uses 30,000 tokens; GPT-2 uses 50,257;
LLaMA-2 uses 32,000.

# Self-Attention: Code Demonstrations

## Scaled Dot-Product Attention in PyTorch

The core attention operation: each token queries every other token for relevant information.


In [ ]:
#| eval: false
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

def scaled_dot_product_attention(Q, K, V):
    """
    Q, K, V: (batch, seq_len, d_k)
    Returns: output (batch, seq_len, d_k), weights (batch, seq_len, seq_len)
    """
    d_k = Q.shape[-1]
    scores = torch.bmm(Q, K.transpose(1, 2)) / (d_k ** 0.5)  # (B, T, T)
    weights = F.softmax(scores, dim=-1)
    output = torch.bmm(weights, V)                              # (B, T, d_k)
    return output, weights

# Demo: 1 batch, 6 tokens, d_k=8
torch.manual_seed(0)
B, T, d_k = 1, 6, 8
Q = torch.randn(B, T, d_k)
K = torch.randn(B, T, d_k)
V = torch.randn(B, T, d_k)

output, attn_weights = scaled_dot_product_attention(Q, K, V)
print(f"Input  shape: {Q.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")

# Visualise the attention map
tokens = ["The", "model", "attends", "to", "each", "token"]
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(attn_weights[0].detach().numpy(), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(T)); ax.set_xticklabels(tokens, rotation=45, ha="right")
ax.set_yticks(range(T)); ax.set_yticklabels(tokens)
ax.set_title("Attention Weight Matrix (single head)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


## Multi-Head Attention with nn.MultiheadAttention

PyTorch's built-in MHA lets us quickly explore how multiple attention heads learn different patterns in a sequence.


In [ ]:
#| eval: false
# Toy sequence: 6 tokens of dimension 16, 2 heads
embed_dim = 16
num_heads  = 2
seq_len    = 6

mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)

torch.manual_seed(42)
x = torch.randn(1, seq_len, embed_dim)           # (batch=1, seq, embed)
attn_out, attn_wt = mha(x, x, x)                  # self-attention: Q=K=V=x

print(f"Output:          {attn_out.shape}")         # (1, 6, 16)
print(f"Attention weight: {attn_wt.shape}")         # (1, 6, 6) averaged over heads

# Heatmap of averaged attention weights
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(attn_wt[0].detach().numpy(), cmap="Oranges")
ax.set_title("MultiheadAttention (heads averaged)")
ax.set_xlabel("Key position"); ax.set_ylabel("Query position")
plt.tight_layout()
plt.show()


## Semantic Similarity with Cosine Distance

Contextual embeddings encode meaning as geometry — semantically similar inputs produce similar vectors.


In [ ]:
#| eval: false
import torch.nn.functional as F

# Simulate contextual embeddings (e.g. from a transformer encoder last layer)
torch.manual_seed(7)
embed_dim = 64

# Three pairs: similar, partially similar, unrelated
phrases = {
    "revenue growth":       torch.randn(embed_dim),
    "sales increase":       None,   # will be constructed as near-copy
    "net income":           torch.randn(embed_dim),
    "operating profit":     None,
    "regulatory risk":      torch.randn(embed_dim),
    "weather forecast":     torch.randn(embed_dim),  # unrelated
}

# Make "sales increase" close to "revenue growth"
phrases["sales increase"]   = phrases["revenue growth"] + 0.1 * torch.randn(embed_dim)
phrases["operating profit"] = phrases["net income"]     + 0.15 * torch.randn(embed_dim)

keys   = list(phrases.keys())
embeds = torch.stack(list(phrases.values()))           # (6, 64)
embeds = F.normalize(embeds, dim=-1)                   # unit-norm

sim_matrix = embeds @ embeds.T                         # cosine similarity matrix

print(f"{'Pair':<40} {'Cosine Sim':>12}")
print("-" * 55)
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        sim = sim_matrix[i, j].item()
        print(f"{keys[i]:<18} ↔  {keys[j]:<18} {sim:>8.3f}")


## What's Next: Pretraining and Adaptation

You now understand how transformers **see** text — through subword tokens — and how their
internal representations encode *meaning* as geometry. The next question is: how do we turn a
generic text-understanding machine into a specialist?

Module 4 covers:

- **Attention in depth** — the Nadaraya–Watson kernel view, additive vs. scaled dot-product
  scoring, and masked softmax for variable-length inputs.
- **Pretraining objectives** — MLM (BERT), CLM (GPT), and span corruption (T5).
- **Adaptation strategies** — from full fine-tuning through instruction tuning, RLHF, and
  parameter-efficient methods (LoRA, Adapters).

### References {.unnumbered}

::: {#refs}
:::
